In [9]:
import glob
import os

import pandas as pd

# Load the dataset
path = r"../datasets/cicids2017"
all_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

# Remove leading and trailing whitespace from column names
df.columns = df.columns.str.strip()

# Display the dataframe info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow IAT Mi

As we can see, there are 79 columns in the dataset. However, there is a duplicate column named `Fwd Header Length.1` which is likely a duplicate of `Fwd Header Length`. We will check if they are identical and then drop the duplicate column if they are indeed the same.

In [10]:
# Check if the duplicate columns are identical, and if so, drop the duplicate column
if df["Fwd Header Length"].equals(df["Fwd Header Length.1"]):
    print(
        "The columns 'Fwd Header Length' and 'Fwd Header Length.1' are identical. Proceeding to drop the duplicate column.")
    df.drop("Fwd Header Length.1", axis=1, inplace=True)
else:
    print("The columns 'Fwd Header Length' and 'Fwd Header Length.1' are not identical.")

The columns 'Fwd Header Length' and 'Fwd Header Length.1' are identical. Proceeding to drop the duplicate column.


Display the dataframe info again to confirm that the duplicate column has been dropped (if they were identical).

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 78 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow IAT Mi

We will convert all columns (except `Label` as it contains categorical data) to numeric values, coercing any errors to NaN. This will help us identify any non-numeric values that may be present in the dataset.

In [12]:
labels = df["Label"]

# Convert all columns (except 'Label') to numeric, coercing errors to NaN
df = df.drop("Label", axis=1).apply(pd.to_numeric, errors="coerce")

# Reattach the 'Label' column to the dataframe
df["Label"] = labels

Check for missing values in the dataset after the conversion to numeric. We will sum the number of missing values in each column and display only those columns that have missing values.

In [13]:
# Check for missing values
missing_values = df.isna().sum()

# Display columns with missing values
missing_values[missing_values > 0]

Flow Bytes/s    1358
dtype: int64

As we can see, missing values are present in the `Flow Bytes/s` column. We will drop any rows that contain missing values in this column to ensure that our dataset is clean and ready for analysis.

In [14]:
# Drop rows with missing values in the 'Flow Bytes/s' column
df.dropna(subset=["Flow Bytes/s"], inplace=True)

# Check for missing values again to confirm that they have been removed
missing_values = df.isna().sum()
missing_values[missing_values > 0]

Series([], dtype: int64)

Now that we have cleaned the dataset by removing duplicate columns and handling missing values, we will encode the `Label` column using Label Encoding. This will convert the categorical labels into numeric values, which can be used for machine learning models.

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Encode the 'Label' column using Label Encoding
df["Label"] = le.fit_transform(df["Label"])

# Create a mapping of the original labels to their encoded values
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

# Display the mapping of original labels to encoded values
print("Label Mapping:")
for original_label, encoded_value in label_mapping.items():
    print(f"\t{original_label}: {encoded_value}")

Label Mapping:
	BENIGN: 0
	Bot: 1
	DDoS: 2
	DoS GoldenEye: 3
	DoS Hulk: 4
	DoS Slowhttptest: 5
	DoS slowloris: 6
	FTP-Patator: 7
	Heartbleed: 8
	Infiltration: 9
	PortScan: 10
	SSH-Patator: 11
	Web Attack � Brute Force: 12
	Web Attack � Sql Injection: 13
	Web Attack � XSS: 14


Lastly, we will save the cleaned dataset to a new CSV file called `cicids2017_cleaned.csv` for future use in our analysis and modeling. Also, we will save the label mapping to a JSON file called `label_mapping.json` to keep track of the original labels and their corresponding encoded values.

In [16]:
# Save the cleaned dataset to a new CSV file
df.to_csv("../datasets/cicids2017_cleaned.csv", index=False)

import json

# Convert the label mapping to a JSON-serializable format and save it to a JSON file
label_mapping = {key: int(value) for key, value in label_mapping.items()}

# Save the label mapping to a JSON file
with open("../datasets/cicids2017_label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)